# ABP Waveform Quality Assessment and Artifact Removal

Exploratory analysis of arterial blood pressure (ABP) waveforms: basic checks, candidate anomaly screening, filtering comparisons, and spectral features.

**Before running:** set `INPUT_CSV`, `ABP_COLUMN`, and `FS`. Do not commit restricted clinical data. Candidate flags and SQI-like features are not clinically validated; a raw waveform is not clean ground truth.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, str(Path.cwd()))
from src.preprocessing import preprocess_all
from src.artifact_detection import abrupt_changes, repeated_value_runs, basic_signal_checks
from src.spectral_analysis import spectral_metrics
from src.signal_quality import amplitude_outlier_fraction, candidate_peak_summary

FS = 62.4725  # Replace with the value from record metadata.
INPUT_CSV = "data/your_abp.csv"  # Update this local path.
ABP_COLUMN = "ABP"


In [ ]:
df = pd.read_csv(INPUT_CSV)
abp_raw = df[ABP_COLUMN].to_numpy(dtype=float)
if len(abp_raw) < 3 or not np.all(np.isfinite(abp_raw)):
    raise ValueError("ABP must contain at least 3 finite samples; handle missing segments explicitly.")
time = np.arange(len(abp_raw)) / FS
print("Samples:", len(abp_raw), "Duration (s):", len(abp_raw)/FS)
print("Basic checks:", basic_signal_checks(abp_raw))
display(pd.Series(abp_raw).describe())


In [ ]:
plt.figure(figsize=(14,4)); plt.plot(time, abp_raw, linewidth=.8)
plt.xlabel("Time (s)"); plt.ylabel("ABP (mmHg)"); plt.title("Raw ABP waveform")
plt.grid(True, alpha=.3); plt.tight_layout(); plt.show()


In [ ]:
# Candidate anomalies for manual review; these are not confirmed artifacts.
display(abrupt_changes(abp_raw, FS, threshold_mmhg=20.0))
display(repeated_value_runs(abp_raw, FS, min_samples=5))


In [ ]:
signals = preprocess_all(abp_raw, fs=FS)
end_time = min(40, len(abp_raw)/FS); start_time = min(30, max(0, end_time-10))
mask = (time >= start_time) & (time <= end_time)
plt.figure(figsize=(14,5))
for name, values in signals.items(): plt.plot(time[mask], values[mask], label=name, linewidth=.9)
plt.xlabel("Time (s)"); plt.ylabel("ABP (mmHg)"); plt.title("Preprocessing comparison")
plt.legend(ncol=2); plt.grid(True, alpha=.3); plt.tight_layout(); plt.show()


In [ ]:
# These metrics quantify change relative to RAW data, not accuracy against a clean reference.
def compare_to_raw(reference, processed):
    error = processed-reference; denom = np.sum((reference-reference.mean())**2)
    return {"RMSE_vs_raw_mmHg": float(np.sqrt(np.mean(error**2))),
            "MAE_vs_raw_mmHg": float(np.mean(np.abs(error))),
            "Correlation_vs_raw": float(np.corrcoef(reference,processed)[0,1]),
            "PRD_vs_raw_percent": float(100*np.sqrt(np.sum(error**2)/denom)) if denom>0 else np.nan}
display(pd.DataFrame({n: compare_to_raw(abp_raw, x) for n,x in signals.items()}).T.round(4))


In [ ]:
display(pd.DataFrame({n: spectral_metrics(x, FS) for n,x in signals.items()}).T.round(4))
print("Exploratory amplitude outlier fraction:", amplitude_outlier_fraction(abp_raw))
print("Candidate peak summary:", {k:v for k,v in candidate_peak_summary(abp_raw,FS).items() if k not in ("peak_indices","peak_prominences_mmhg")})
